<a href="https://colab.research.google.com/github/KhushiKeswani/MyVectorDB/blob/main/MyVectorDB.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#linux command to download zipped data in runtime
!wget -q https://www.dropbox.com/s/vs6ocyvpzzncvwh/new_articles.zip

In [2]:
#unzip zipped data
!unzip /content/new_articles.zip -d /content/drive/MyDrive/new_articles

Archive:  /content/new_articles.zip
replace /content/drive/MyDrive/new_articles/05-07-fintech-space-continues-to-be-competitive-and-drama-filled.txt? [y]es, [n]o, [A]ll, [N]one, [r]ename: 

In [4]:
import os

In [5]:
#merging content of different text files present in directory, to a single text file
source_dir = '/content/drive/MyDrive/new_articles'
output_dir = '/content/drive/MyDrive/merged_articles_content.txt'
files = [f for f in os.listdir(source_dir) if f.endswith('.txt')]
for file in files:
  file_path = os.path.join(source_dir, file)
  with open(file_path, 'r') as f:
    content = f.read()
  with open(output_dir, 'a') as f:
    f.write(content)

In [1]:
output_dir = '/content/drive/MyDrive/merged_articles_content.txt'

In [2]:
#converting raw text to small sized chunks
with open(output_dir, 'r') as f:
  content = f.read()
def chunk_text(text,chunk_size= 200):
  words = text.split()
  chunks = []
  for i in range(len(words)-chunk_size):
    chunk = ' '.join(words[i:i+chunk_size])
    chunks.append(chunk)
  return chunks
chunks = chunk_text(content)

In [10]:
chunks[0:5]

['Welcome to The Interchange! If you received this in your inbox, thank you for signing up and your vote of confidence. If you’re reading this as a post on our site, sign up here so you can receive it directly in the future. Every week, we’ll take a look at the hottest fintech news of the previous week. This will include everything from funding rounds to trends to an analysis of a particular space to hot takes on a particular company or phenomenon. There’s a lot of fintech news out there and it’s our job to stay on top of it — and make sense of it — so you can stay in the know. — Mary Ann and Christine Busy, busy, busy It was a busy week in startup and venture lands, and the fintech space was no exception. In the venture world, I reported on Peter Ackerson’s departure from Fin Capital earlier this year and the fact that he has since started a new venture firm called Audere Capital. The circumstances around his departure remain fuzzy, but one source speculated that tension arose between

In [4]:
#using pre trained transformer for embedding chunks of data
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(
    'all-MiniLM-L6-v2'
)

embedding = model.encode(chunks)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [12]:
embedding[0:5]

array([[-0.05279564, -0.12433624, -0.00276254, ..., -0.11289125,
        -0.01091943,  0.00681925],
       [-0.04978026, -0.12087627, -0.00184563, ..., -0.11149438,
         0.00106298,  0.0010008 ],
       [-0.05393552, -0.12867293, -0.00726829, ..., -0.10883208,
         0.00839088,  0.00630105],
       [-0.05186214, -0.12447664, -0.01020243, ..., -0.10159786,
         0.00803297,  0.00650516],
       [-0.04680544, -0.12214423, -0.00817147, ..., -0.10789344,
         0.01295997,  0.00766275]], dtype=float32)

In [13]:
embedding.shape

(27658, 384)

In [5]:
#mapping chunk data to index
doc_store = {
    i: chunks[i]
    for i in range(len(chunks))
}

In [15]:
doc_store[5]

'you received this in your inbox, thank you for signing up and your vote of confidence. If you’re reading this as a post on our site, sign up here so you can receive it directly in the future. Every week, we’ll take a look at the hottest fintech news of the previous week. This will include everything from funding rounds to trends to an analysis of a particular space to hot takes on a particular company or phenomenon. There’s a lot of fintech news out there and it’s our job to stay on top of it — and make sense of it — so you can stay in the know. — Mary Ann and Christine Busy, busy, busy It was a busy week in startup and venture lands, and the fintech space was no exception. In the venture world, I reported on Peter Ackerson’s departure from Fin Capital earlier this year and the fact that he has since started a new venture firm called Audere Capital. The circumstances around his departure remain fuzzy, but one source speculated that tension arose between Ackerson and Fin founding partn

In [6]:
!pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 86.1 MB/s eta 0:00:00


In [7]:
#implementing chunk indexing, using HNSW algorithm which helps in efficient fast search among chunks
import faiss
import numpy as np

embeddings = np.array(
    embedding
).astype('float32')

dimension = embeddings.shape[1]
index = faiss.IndexHNSWFlat(
    dimension, 32
)

index.add(embeddings)

In [33]:
#query embedding, user query is also embedded into numerical features
query_embedding = model.encode(
    ["What is Google Pixel 7a"]
)

In [34]:
#computing distance between chunks and user query, top 3 chunk's index and distance are shown
distances, indices = index.search(
    np.array(query_embedding).astype('float32'),
    k=3
)

In [35]:
distances

array([[0.5611044 , 0.5611044 , 0.56525826]], dtype=float32)

In [36]:
indices

array([[16114, 42648, 16024]])

In [37]:
#retrieving top 3 similar data chunks
results = []

for idx in indices[0]:

    results.append(
        doc_store[idx]
    )

print(results)

['It’s a good system that works, and Google’s newly focused mobile hardware team has created some surprisingly good devices at extremely reasonable prices. Never one to be outdone by the deluge of rumors, the company went ahead and announced via Twitter its next device is due out on May 11 — the day after Google I/O and, perhaps not coincidentally, my birthday. It was Google India that specifically made the announcement — perhaps not surprising, as the company is likely to aggressively target the world’s number one smartphone market with the product. The image points to a very similar design as the 7 — not really a surprise as these things go. Though it does stop short of actually mentioning the name, as it’s done in the past. Basically expect the 7 with cheaper materials. Rumors point to a 6.1-inch device featuring a 90Hz refresh rate, coupled with a 64-megapixel rear camera. The 7’s Tensor G2 returns for a command performance, likely bringing with it many of the software features it 

In [15]:
#building user defined function for whole search flow
def search(query, k=3):

    query_embedding = model.encode([query])

    distances, indices = index.search(
        np.array(query_embedding).astype('float32'),
        k
    )

    results = []

    for idx in indices[0]:

        results.append(doc_store[idx])

    return results

In [38]:
search('What is Google I/O?')

['The site’s current search tool uses only approximate matches between text and images as well as URL searches to find content hosted on specific websites. Spawning — now beholden to investors — plans to make money by building services on top of its content infrastructure, although Meyer wouldn’t divulge much. How that’ll sit with content creators remains to be seen. “We’ve spoken to quite a few organizations, with many conversations being too premature to announce, and think that our funding announcement and increased visibility will go some way to offer assurances that what we are building is a robust and dependable standard to work with,” Meyer said. “After we complete these features, we’ll begin building infrastructure to support more datasets — including music, video and text.”After Google cut all but three of the projects at its in-house incubator Area 120 and shifted it to work on AI projects across Google, one of the legacy efforts — coincidentally also an AI project — is now o

In [44]:
query = 'What is news about Databricks?'

In [45]:
#reranking model's output with crossencoder, that compares query with each output and gives score
from sentence_transformers import CrossEncoder

reranker = CrossEncoder(
    'cross-encoder/ms-marco-MiniLM-L-6-v2'
)

pairs = [
    [query, chunk]
    for chunk in search(query)
]

scores = reranker.predict(pairs)

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [46]:
scores

array([0.6278433 , 0.81905127, 0.66948867], dtype=float32)

In [49]:
search('what is news about databricks?')

['so it seems that they remain in a holding pattern. (We’ll update this if and when we hear more.) In the meantime, Area 120 itself is also seeing some revolving doors. Clay Bavor, who was running Area 120 among other things and who messaged the big changes to staff in January, was out the door just a month later. He has now teamed up with Bret Taylor — another ex-Googler who has an outsized track record that includes being the CTO of Facebook and the co-CEO of Salesforce — to work on a mystery startup. Updated with more information about Checks’ valuation and quote from Google.Databricks today announced that it has acquired Okera, a data governance platform with a focus on AI. The two companies did not disclose the purchase price. According to Crunchbase, Okera previously raised just under $30 million. Investors include Felicis, Bessemer Venture Partners, Cyber Mentor Fund, ClearSky and Emergent Ventures. Data governance was already a hot topic, but the recent focus on AI has highligh

In [50]:
#storing embedding in a numpy file
np.save(
    "/content/drive/MyDrive/myvectordb/embeddings.npy",
    embedding
)

In [51]:
#saving chunks in a file
import json
with open("/content/drive/MyDrive/myvectordb/doc_store.json","w") as f:
  json.dump(doc_store,f)

In [52]:
#save FAISS index
faiss.write_index(
    index,
    "/content/drive/MyDrive/myvectordb/vector.index"
)

We have built a persistent VectorDB, enabling vector search and efficient retrieval with reranking